# Value-at-Risk of a Ruble-Based Equity Portfolio Through the 2022 Shock

A Russian investor holds a portfolio of German and US stocks, measured in rubles: 40% tracks the DAX with beta 1.3, and 60% tracks the NASDAQ-100 with beta 1.6. The portfolio is therefore exposed to two risk factors, equity and EUR/RUB-USD/RUB foreign exchange.

The notebook measures how its risk changed between **23 February 2022**, the day before Russia invaded Ukraine, and **9 March 2022**, two weeks into the ruble collapse, using:

1. EWMA volatilities and correlation (RiskMetrics, λ = 0.94),
2. Normal parametric VaR, systematic and stand-alone, at 1 and 10 days,
3. Historical VaR with equal weights, exponential age weights (Boudoukh, Richardson and Whitelaw) and volatility weighting (Hull and White).

All VaR figures are at 99% (α = 1%) and expressed as a fraction of portfolio value.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import norm

sns.set_theme(style="whitegrid")

WEIGHTS = np.array([0.4, 0.6])          # DAX, NASDAQ-100
BETAS = np.array([1.3, 1.6])
LAMBDA = 0.94
ALPHA = 0.01
TRADING_DAYS = 250
DATES = {"23 Feb 2022": "2022-02-23", "9 Mar 2022": "2022-03-09"}

## Data

Daily closes from Yahoo Finance: DAX (`^GDAXI`), NASDAQ-100 (`^NDX`), EUR/RUB and USD/RUB. The first run downloads them; the snapshot in `data/` keeps later runs identical.

In [ ]:
path = Path("data/market_data.csv")
if not path.exists():
    import yfinance as yf
    tickers = {"^GDAXI": "DAX", "^NDX": "NASDAQ100", "EURRUB=X": "EUR/RUB", "RUB=X": "USD/RUB"}
    raw = yf.download(list(tickers), start="2019-03-01", end="2022-03-10", auto_adjust=False, progress=False)["Close"]
    path.parent.mkdir(exist_ok=True)
    raw.rename(columns=tickers).to_csv(path)

px = pd.read_csv(path, index_col=0, parse_dates=True)
# Equity trading days drive the calendar; FX gaps are filled with the last quote.
px[["EUR/RUB", "USD/RUB"]] = px[["EUR/RUB", "USD/RUB"]].ffill()
px = px.dropna(subset=["DAX", "NASDAQ100"]).dropna()
print(px.shape, px.index.min().date(), "to", px.index.max().date())
px.tail()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
px[["DAX", "NASDAQ100"]].plot(ax=axes[0], title="Indices in local currency")
px[["EUR/RUB", "USD/RUB"]].plot(ax=axes[1], title="Rubles per euro and per dollar")
rub = pd.DataFrame({"DAX in RUB": px["DAX"] * px["EUR/RUB"], "NASDAQ-100 in RUB": px["NASDAQ100"] * px["USD/RUB"]})
(rub / rub.iloc[0]).plot(ax=axes[2], title="Indices in rubles (start = 1)")
for ax in axes:
    ax.axvline(pd.Timestamp("2022-02-24"), color="grey", ls="--", lw=1)
plt.tight_layout()
plt.show()

In local currency both indices dip in March 2020 and again in early 2022. In rubles the picture flips after 24 February: the currency collapse lifts the ruble value of foreign holdings even as the indices fall.

## Portfolio returns

Equity return $= \sum_i w_i \beta_i\, y_{i,t}$ and forex return $= \sum_i w_i\, x_{i,t}$, with log returns $y$ for the indices and $x$ for the exchange rates. The ruble return of the portfolio is their sum.

In [ ]:
logret = np.log(px).diff().dropna()
returns = pd.DataFrame({
    "equity": logret[["DAX", "NASDAQ100"]].to_numpy() @ (WEIGHTS * BETAS),
    "forex": logret[["EUR/RUB", "USD/RUB"]].to_numpy() @ WEIGHTS,
}, index=logret.index)
returns["portfolio"] = returns["equity"] + returns["forex"]

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
returns[["equity", "forex"]].plot(ax=axes[0], lw=0.8, title="Daily equity and forex returns")
sns.histplot(returns["portfolio"], bins=100, ax=axes[1]).set_title("Daily portfolio returns")
plt.tight_layout()
plt.show()
returns.describe().T[["mean", "std", "min", "max"]].assign(
    skew=returns.skew(), excess_kurtosis=returns.kurt())

The portfolio return distribution is centred near zero but far from normal: heavy tails (large excess kurtosis), driven by March 2020 and the 2022 ruble moves.

## EWMA volatilities and correlation

$\hat\sigma^2_t = (1-\lambda)\, r^2_{t-1} + \lambda\, \hat\sigma^2_{t-1}$, and the same recursion for the covariance, seeded with the full-sample mean square.

In [ ]:
def ewma(x, y=None, lam=LAMBDA):
    """EWMA variance of x (or covariance of x and y), using returns up to t-1."""
    y = x if y is None else y
    prod = (x * y).to_numpy()
    out = np.empty_like(prod)
    out[0] = prod.mean()
    for t in range(1, len(prod)):
        out[t] = (1 - lam) * prod[t - 1] + lam * out[t - 1]
    return pd.Series(out, index=x.index)

ew = pd.DataFrame({
    "var_equity": ewma(returns["equity"]),
    "var_forex": ewma(returns["forex"]),
    "cov": ewma(returns["equity"], returns["forex"]),
    "var_portfolio": ewma(returns["portfolio"]),
})
ew["vol_equity"], ew["vol_forex"] = np.sqrt(ew["var_equity"]), np.sqrt(ew["var_forex"])
ew["corr"] = ew["cov"] / (ew["vol_equity"] * ew["vol_forex"])

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
(np.sqrt(TRADING_DAYS) * ew[["vol_equity", "vol_forex"]]).plot(ax=axes[0], title="EWMA volatility (annualised)")
ew["corr"].plot(ax=axes[1], title="EWMA equity-forex correlation")
plt.tight_layout()
plt.show()

## EWMA annual covariance matrices

In [ ]:
def annual_cov(date):
    row = ew.loc[date]
    return TRADING_DAYS * np.array([[row["var_equity"], row["cov"]], [row["cov"], row["var_forex"]]])

cov = {label: annual_cov(d) for label, d in DATES.items()}
for label, m in cov.items():
    print(label); print(pd.DataFrame(m, index=["equity", "forex"], columns=["equity", "forex"]).round(4), "\n")

## Normal parametric VaR

$\text{VaR}_{h,\alpha} = \Phi^{-1}(1-\alpha)\sqrt{\tfrac{h}{250}\,\theta^\top \Sigma\, \theta}$ with annual covariance $\Sigma$. The systematic VaR uses $\theta = (1, 1)$; the stand-alone equity and forex VaRs use $(1, 0)$ and $(0, 1)$.

In [ ]:
def normal_var(theta, sigma, days):
    theta = np.asarray(theta, dtype=float)
    return norm.ppf(1 - ALPHA) * np.sqrt(days / TRADING_DAYS * theta @ sigma @ theta)

parametric = pd.DataFrame({
    label: {
        "Systematic, 1 day": normal_var([1, 1], m, 1),
        "Systematic, 10 days": normal_var([1, 1], m, 10),
        "Equity stand-alone, 10 days": normal_var([1, 0], m, 10),
        "Forex stand-alone, 10 days": normal_var([0, 1], m, 10),
    } for label, m in cov.items()})
parametric["change"] = parametric["9 Mar 2022"] / parametric["23 Feb 2022"]
parametric.round(4)

## Historical VaR

Three weightings of the same portfolio return history, each using data up to the valuation date only:

- **Equal weights:** the plain 1% empirical quantile.
- **Age weights (BRW):** the observation $k$ days old gets weight $\propto \lambda^{k}$; VaR is the return at which the weights of the sorted losses first reach 1%.
- **Volatility weights (Hull-White):** each return is rescaled by $\hat\sigma_T / \hat\sigma_t$ so that history is expressed at today's volatility.

In [ ]:
def weighted_quantile(values, weights, q):
    """Smallest value whose cumulative weight (sorted ascending) reaches q."""
    order = np.argsort(values)
    cum = np.cumsum(np.asarray(weights)[order]) / np.sum(weights)
    return np.asarray(values)[order][np.searchsorted(cum, q)]

def historical_vars(date):
    r = returns.loc[:date, "portfolio"]
    age = np.arange(len(r))[::-1]                         # 0 for the valuation date
    sigma = np.sqrt(ew.loc[:date, "var_portfolio"])
    vol_weighted = r * sigma.iloc[-1] / sigma
    return {
        "Equal weights": -np.quantile(r, ALPHA),
        "Age weights (BRW)": -weighted_quantile(r.to_numpy(), LAMBDA ** age, ALPHA),
        "Volatility weights (Hull-White)": -np.quantile(vol_weighted, ALPHA),
    }

historical = pd.DataFrame({label: historical_vars(d) for label, d in DATES.items()})
historical["change"] = historical["9 Mar 2022"] / historical["23 Feb 2022"]
historical.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for ax, (label, d) in zip(axes, DATES.items()):
    r = returns.loc[:d, "portfolio"]
    sns.histplot(r, bins=80, ax=ax, stat="density", alpha=0.4)
    for name, v in historical[label].items():
        ax.axvline(-v, ls="--", label=f"{name}: {v:.3f}")
    ax.set_title(f"Portfolio returns up to {label}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Summary

In [ ]:
summary = pd.concat([parametric.drop(columns="change").loc[["Systematic, 1 day"]].rename(index=lambda s: "Normal parametric"),
                     historical.drop(columns="change")])
summary["change"] = summary["9 Mar 2022"] / summary["23 Feb 2022"]
summary.round(4)

In [ ]:
# Checks on the maths.
assert np.isclose(weighted_quantile(np.array([3., 1., 2.]), np.ones(3), 0.34), 2.0)
assert np.isclose(weighted_quantile(np.array([-5., 0., 1.]), np.array([0.02, 0.49, 0.49]), ALPHA), -5.0)
for label in DATES:
    p = parametric[label]
    assert np.isclose(p["Systematic, 10 days"], np.sqrt(10) * p["Systematic, 1 day"])
    assert p["Systematic, 10 days"] <= p["Equity stand-alone, 10 days"] + p["Forex stand-alone, 10 days"] + 1e-12
assert (historical.drop(columns="change") > 0).all().all()

## Conclusion

CONCLUSION_PLACEHOLDER